# Policy Gradient Methods

Policy Gradient (PG) methods directly optimize a parameterized stochastic policy $\pi_{\mathbf{\theta}}(a|s)$ to maximize the expected return. Unlike value-based methods that derive a policy implicitly from value estimates, PG methods compute an estimator of the gradient of the expected return and perform gradient ascent directly on the policy parameters $\mathbf{\theta}$.

Let the expected return be defined as:
$$
J(\mathbf{\theta}) = \mathbb{E}_{\tau \sim \pi_{\mathbf{\theta}}}[R(\tau)]
$$

where $\tau = (s_0, a_0, s_1, a_1, \dots)$ represents a trajectory, and $R(\tau)$ is the cumulative reward. The foundational [Policy Gradient Theorem](https://proceedings.neurips.cc/paper_files/paper/1999/file/464d828b85b0bed98e80ade0a5c43b0f-Paper.pdf) establishes that the gradient of this objective with respect to the policy parameters can be expressed as an expectation over trajectories sampled from the current policy:
$$
\nabla_{\mathbf{\theta}} J(\mathbf{\theta}) = \mathbb{E}_{\tau \sim \pi_{\mathbf{\theta}}} \left[ \sum_{t=0}^{H-1} \nabla_{\mathbf{\theta}} \log \pi_{\mathbf{\theta}}(a_t | s_t) R(\tau) \right]
$$

This formulation allows us to estimate the gradient using sample trajectories. The general optimization procedure is iterative: at each iteration, the agent interacts with the environment to collect data, estimates the policy gradient, and updates the parameters using stochastic gradient ascent. This generic training loop is outlined in Algorithm 1.

**Algorithm 1** Policy Gradient (PG)
***
**Input:** MDP $\mathcal{M}$, number of iterations $T$, learning rate schedule $(\eta^{(t)})_{t=0}^{T-1}$<br>
**Output:** approximately optimal policy parameters $\mathbf{\theta}^{(T)}$

1. Initialize $\mathbf{\theta}^{(0)}$ arbitrarily
2. **for** $t = 0, 1, \dots, T - 1$ **do**
    * Estimate the policy gradient $\widehat{\nabla}_{\mathbf{\theta}} J(\mathbf{\theta}^{(t)})$
    * Update the parameters $\mathbf{\theta}^{(t+1)} = \mathbf{\theta}^{(t)} + \eta^{(t)} \widehat{\nabla}_{\mathbf{\theta}} J(\mathbf{\theta}^{(t)})$
3. **end for**
4. **return** $\mathbf{\theta}^{(T)}$
***

*(Note: In the algorithm above, $t$ denotes the outer training iteration. In the subsequent derivations, $t$ will denote the time step within a specific environment trajectory, while $H$ will represent the trajectory horizon).*

# REINFORCE

In this section, we implement the [REINFORCE](https://link.springer.com/content/pdf/10.1007/BF00992696.pdf) algorithm utilizing the low-variance estimator provided in [Baxter et al. 2001](https://arxiv.org/pdf/1106.0665), often associated with the GPOMDP formulation. This method aims to optimize policies in RL by estimating the gradient of the expected reward with respect to policy parameters. First, we provide the mathematical formulation of the gradient estimator, highlighting its derivation. Following this, we provide its implementation in Python.

While standard REINFORCE provides an unbiased estimate of the policy gradient, it suffers from notoriously high variance. This variance originates from using the full trajectory return, $R(\tau) = \sum_{t=0}^{H-1} \gamma^t r(s_t, a_t)$, to weight the score function of every action within that trajectory. Because $R(\tau)$ aggregates stochasticity from environment dynamics, reward functions, and the policy itself over the entire horizon $H$, it acts as a highly noisy baseline. Furthermore, this naive formulation violates temporal causality: an action $a_t$ is weighted by rewards accumulated *before* $t$. Assigning credit to actions for outcomes they could not possibly have influenced injects noise, substantially inflating the variance of the estimator.

To address this, we rely on the principle that an action $a_t$ taken at time $t$ can only influence future rewards. We substitute the full episode return with the **reward-to-go**, defined as the discounted sum of rewards received strictly after time step $t$:
$$
\hat{R}_t = \sum_{t'=t}^{H-1} \gamma^{t'} r(s_{t'}, a_{t'})
$$

By incorporating the reward-to-go, the variance of the gradient estimator is significantly reduced without introducing bias. The expected gradient of the objective function is thus formulated as:
$$
\nabla_{\mathbf{\theta}} J(\mathbf{\theta}) \approx \frac{1}{N} \sum_{i=1}^N \sum_{t=0}^{H-1} \nabla_{\mathbf{\theta}} \log \pi_{\mathbf{\theta}}(a_{i,t} | s_{i,t}) \hat{R}_{i,t}
$$

An algebraically equivalent sequence, which is highly efficient for vectorized environments and deep learning frameworks, computes the cumulative sum of the score functions multiplied by the discounted rewards at each time step:
$$
\nabla_{\mathbf{\theta}}^{\text{GPOMDP}} J(\mathbf{\theta}) = \frac{1}{N} \sum_{i=1}^N \sum_{t=0}^{H-1} \gamma^t r(s_{i,t}, a_{i,t}) \sum_{l=0}^t \nabla_{\mathbf{\theta}} \log \pi_{\mathbf{\theta}}(a_{i,l} | s_{i,l})
$$

The learning procedure follows these standard steps:
* Initialize the policy parameter $\mathbf{\theta}$.
* Generate a batch of $N$ trajectories on policy $\pi_{\mathbf{\theta}}$.
* For each time step $t$ in the sequence, estimate the gradient using the reward-to-go calculation shown above.
* Update the policy parameters using gradient ascent:
$$
\mathbf{\theta} \leftarrow \mathbf{\theta} + \alpha \widehat{\nabla}_{\mathbf{\theta}}^{\text{GPOMDP}} J(\mathbf{\theta})
$$

This optimization strategy allows the network to isolate the direct consequences of its actions, leading to more stable and efficient convergence compared to standard sequence-level reinforcement algorithms.

### ⭐ Exercise

In the next cell, implement the GPOMDP estimator for linear Gaussian policies based on the formal definitions provided above. 

Hint: You can leverage the `torch.cumsum` mathematical operation across the temporal axis to efficiently aggregate the score functions $\nabla_{\mathbf{\theta}} \log \pi_{\mathbf{\theta}}(a | s)$ before computing their dot product with the discounted rewards.

In [ ]:
def reinforce_linear(rollouts, policy, gamma):
    """
    Computes the policy gradient using the REINFORCE (reward-to-go) estimator.
    """
    grad = 0

    for roll in rollouts:
        H = len(roll)
        disc_rew = torch.zeros((H, 1))
        scores = torch.zeros((H, policy.dim))

        # Retrieve the score of the policy and compute the discounted rewards
        for t in range(H):
            s, a, r = roll[t]
            disc_rew[t] = (gamma ** t) * r
            scores[t] = policy.grad_log(s, a)

        # Aggregate the score functions up to time t
        cum_scores = torch.cumsum(scores, dim=0)

        # Multiply by the discounted reward at time t and sum over the horizon
        grad += torch.sum(cum_scores * disc_rew, dim=0)

    return grad / len(rollouts)

### ⭐ Exercise
 
Re-implement the GPOMDP estimator so that it works for **any** neural network policy, without manually computing the score function. Instead of calling `policy.grad_log()`, build a scalar **surrogate loss** whose gradient w.r.t. $\mathbf{\theta}$ is exactly the GPOMDP estimator:
 
$$\mathcal{L}(\mathbf{\theta}) = \frac{1}{N} \sum_{i=1}^N \sum_{t=0}^{H-1} \hat{R}_{i,t} \cdot \log \pi_{\mathbf{\theta}}(a_{i,t} \mid s_{i,t})$$
 
where $\hat{R}_{i,t} = \sum_{t'=t}^{H-1} \gamma^{t'} r(s_{i,t'}, a_{i,t'})$ is the discounted reward-to-go, then call `.backward()` to let autograd handle the rest.
 
**Hints:**
- Call `policy.zero_grad()` at the start to clear stale gradients.
- To compute the rewards-to-go $\hat{R}_{i,t}$ efficiently, combine `torch.flip` and `torch.cumsum` (same trick as before, but reversed).
- Add a small numerical guard: `torch.log(action_probs[a_t] + 1e-8)`.
- After `.backward()`, extract the flat gradient with `nn.utils.parameters_to_vector([p.grad for p in policy.parameters()])`.


In [ ]:
def reinforce(rollouts, policy, gamma):
    """
    Computes the policy gradient using the REINFORCE (reward-to-go) gradient estimator using PyTorch autograd.
    """
    policy.zero_grad()
    total_loss = torch.tensor(0.0)

    for roll in rollouts:
        H = len(roll)

        # Compute discounted rewards-to-go
        disc_rewards = torch.tensor(
            [gamma**t * roll[t][2] for t in range(H)], dtype=torch.float32
        )

        rewards_to_go = torch.flip(
            torch.cumsum(torch.flip(disc_rewards, dims=[0]), dim=0), dims=[0]
        )

        # Accumulate the surrogate loss
        for l in range(H):
            s_l, a_l, _ = roll[l]
            state_tensor = torch.FloatTensor(s_l)

            # Forward pass — works for any architecture
            action_probs = policy(state_tensor)

            log_prob = torch.log(action_probs[a_l] + 1e-8)

            
            total_loss = total_loss + rewards_to_go[l] * log_prob

    # Average over rollouts
    total_loss = total_loss / len(rollouts)

    # Single backward pass computes all gradients
    total_loss.backward()

    # Return flat gradient vector
    grad = nn.utils.parameters_to_vector(
        [p.grad for p in policy.parameters()]
    ).detach()

    return grad

# Baselines
 
A well-known technique to further reduce the variance of the REINFORCE estimator is to subtract a **baseline** $b$ from the reward-to-go. The gradient estimator becomes:
 
$$
\nabla_{\mathbf{\theta}} J(\mathbf{\theta}) \approx \frac{1}{N} \sum_{i=1}^N \sum_{t=0}^{H-1} \nabla_{\mathbf{\theta}} \log \pi_{\mathbf{\theta}}(a_{i,t} | s_{i,t}) \left(\hat{R}_{i,t} - b\right)
$$
 
This modification is **unbiased** as long as $b$ does not depend on the action $a_{i,t}$. This follows directly from the log-derivative trick:
 
$$
\mathbb{E}_{\pi_{\mathbf{\theta}}}\left[\nabla_{\mathbf{\theta}} \log \pi_{\mathbf{\theta}}(a_t | s_t) \cdot b \right] = b \nabla_{\mathbf{\theta}} \underbrace{\sum_a \pi_{\mathbf{\theta}}(a | s_t)}_{=1} = 0
$$
 
so subtracting $b$ leaves the expectation of the estimator unchanged while reducing its variance.
 
Two common choices are:
 
- **Constant baseline.** A fixed scalar $b = c$, typically set to the average return observed over a long training window. Simple to implement, but it does not adapt to the current policy.
- **Average baseline.** The mean reward-to-go computed over the current batch of trajectories:
$$
b = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{H} \sum_{t=0}^{H-1} \hat{R}_{i,t}
$$
This adapts automatically at each iteration and is the most common choice in practice, as it centers the rewards within each batch and reduces variance without requiring any additional hyperparameters.
### ⭐ Exercise
 
Extend `reinforce` to support the **average baseline**. Before accumulating the surrogate loss, compute $b$ as the mean of all rewards-to-go across the batch, then subtract it from each $\hat{R}_{i,t}$.
 
**Hints:**
- Collect all `rewards_to_go` tensors across rollouts into a list, then use `torch.cat` and `torch.mean` to compute $b$ before the main accumulation loop.
- Everything else stays the same — the only change is replacing `rewards_to_go[t]` with `rewards_to_go[t] - b`.


In [ ]:
def reinforce_baseline(rollouts, policy, gamma):
    """
        Computes the policy gradient using the REINFORCE (reward-to-go) gradient estimator with average baseline.
    """
    policy.zero_grad()
    total_loss = torch.tensor(0.0)
 
    # Pre-compute rewards-to-go for all rollouts
    all_rewards_to_go = []
    for roll in rollouts:
        H = len(roll)
        disc_rewards = torch.tensor(
            [gamma**t * roll[t][2] for t in range(H)], dtype=torch.float32
        )
        rewards_to_go = torch.flip(
            torch.cumsum(torch.flip(disc_rewards, dims=[0]), dim=0), dims=[0]
        )
        all_rewards_to_go.append(rewards_to_go)
 
    # Compute the average baseline
    b = torch.mean(torch.cat(all_rewards_to_go))
 
    # Accumulate the surrogate loss with baseline
    for roll, rewards_to_go in zip(rollouts, all_rewards_to_go):
        H = len(roll)
        for t in range(H):
            s_t, a_t, _ = roll[t]
            action_probs = policy(torch.FloatTensor(s_t))
            log_prob = torch.log(action_probs[a_t] + 1e-8)
            total_loss = total_loss + (rewards_to_go[t] - b) * log_prob
 
    total_loss = total_loss / len(rollouts)
    total_loss.backward()
 
    grad = nn.utils.parameters_to_vector(
        [p.grad for p in policy.parameters()]
    ).detach()
 
    return grad


# FARE VERSIONE GPOMDP LINEAR. usare nomi diversi per le performance etc. aggiungere baselines (average). nella toria solo constant and average

# Training loop

Finally, let's run the learning loop.
As the structure of the agent is very similar to Q-learning agent, the learning loop is very similar as well. Notice that we perform a learning step on a batch of samples instead of a single datapoint.

In [10]:

def train(env, policy, gamma, m, K, alpha, T, estimator):
    """
    Train a policy with G(PO)MDP
    :param env: (Env object) the Gym environment
    :param policy: (Policy object) the policy
    :param gamma: (float) the discount factor
    :param m: (int) number of episodes per iterations
    :param K: (int) maximum number of iterations
    :param alpha: (float) the constant learning rate
    :param T: (int) the trajectory horizon
    :return: list (ndarray, ndarray) the evaluations
    """

    results = []

    # Evaluate the initial policy
    res = evaluate(env, policy, gamma)
    results.append(res)

    for k in range(K):

        print('Iteration:', k)

        # Generate rollouts
        rollouts = collect_rollouts(env, policy, m, T)

        # Get policy parameter
        theta = policy.get_parameters()

        # Call your Gradient estimator
        if estimator == 'REINFORCE':
          pg = reinforce(rollouts, policy, gamma)
        else:
          pg = gpomdp(rollouts, policy, gamma)

        # Update policy parameter
        theta = theta + alpha * pg

        # Set policy parameters
        policy.set_parameters(theta)

        # Evaluate the updated policy
        res = evaluate(env, policy, gamma)
        results.append(res)

    return results

## ⭐ Exercise

- Choose the parameters (number of training episodes, numbers of hidden units in MLP, learning rate, batch size) so that the average return of the agent increases with training and ends up being greater than $100$.
- Run the learning loop and visualise the epsiode returns. Look at the animation of the last episode. Optional: modify the code and try both gradient estimators. What can you notice?

In [11]:
discount_factor = 0.999 # @param
batch_size = 100
iterations = 100 # @param
learning_rate = 0.001 # @param
trajectory_length = 200 # @param
n_hidden = 8 # @param

# gradient estimator {'REINFORCE', 'gpomdp'}
estimator = 'gpomdp' # @param

# Instantiate the policy
policy = SoftmaxPolicyNetwork(s_size, a_size, n_hidden)

# Start training
results = train(env, policy, discount_factor, batch_size, iterations, learning_rate, trajectory_length, estimator)

Mean reward: 24.2958939929817 Std reward: 1.3519187285432552 Num episodes: 100
Iteration: 0
Mean reward: 23.573139927965634 Std reward: 1.1278473174116315 Num episodes: 100
Iteration: 1
Mean reward: 26.870987126203822 Std reward: 1.472281307075575 Num episodes: 100
Iteration: 2
Mean reward: 25.72028358618262 Std reward: 1.6003275948836695 Num episodes: 100
Iteration: 3
Mean reward: 23.870005812601747 Std reward: 1.0936860109994746 Num episodes: 100
Iteration: 4
Mean reward: 23.21676585446906 Std reward: 1.165932861391778 Num episodes: 100
Iteration: 5
Mean reward: 26.30605787667935 Std reward: 1.5959926029577103 Num episodes: 100
Iteration: 6


KeyboardInterrupt: 

In [ ]:
# Plot the results of the training
plot_results(results)

In [ ]:
# Evaluate the policy and animate the results
perf_mean, perf_std, frames = evaluate(eval_env, policy, num_episodes=1)

# Animate the last episode
animate(frames)

# 🥳 Congratulations on completing the second part of this tutorial, great job!!!

In this part you learnt how policy gradient works and how to combine it with neural networks. Next, we will look into RL from Human Feedback (RLHF).